[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IyadSultan/CCI/blob/main/session5/solutions/Lab1_LangChain_Foundations_Solutions.ipynb)

# Lab 1: LangChain v1 Foundations — SOLUTIONS
## CCI Session 5

**Duration:** 10 minutes

### Clinical Scenario
> You're building a clinical assistant at KHCC. In Session 3 you did this with raw OpenAI API — now you'll see how LangChain simplifies the same task.

### Objective
- Set up LangChain with ChatOpenAI
- Compare raw OpenAI API vs LangChain approach
- Use `create_react_agent` to build a basic agent
- Understand Runnables and composition

---
## Setup

In [ ]:
# Install required packages
!pip install -q langchain langchain-openai langgraph

In [ ]:
import os
from google.colab import userdata

# Load your OpenAI API key from Colab secrets
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
print("API key loaded.")

---
## Cell 1 — ChatOpenAI Basics

In Session 3, you used the raw OpenAI client:
```python
from openai import OpenAI
client = OpenAI()
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Hello"}]
)
print(response.choices[0].message.content)
```

LangChain wraps this with `ChatOpenAI`, giving you a consistent interface, streaming, callbacks, and composability.

In [ ]:
from langchain_openai import ChatOpenAI

# SOLUTION: Create a ChatOpenAI instance
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# SOLUTION: Call the LLM with a simple clinical question
response = llm.invoke("What are the first-line treatments for chronic myeloid leukemia?")

print("Type:", type(response))
print("Content:", response.content)

**Key difference from Session 3:** Notice `response.content` directly gives you the text — no need for `response.choices[0].message.content`.

---
## Cell 2 — Messages and Prompt Templates

In Session 3, you built messages as plain dicts:
```python
messages = [
    {"role": "system", "content": "You are a clinical assistant."},
    {"role": "user", "content": "What is metformin used for?"}
]
```

LangChain provides typed message classes and **prompt templates** that make prompts reusable.

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.prompts import ChatPromptTemplate

# --- Approach A: Direct message objects ---
messages = [
    SystemMessage(content="You are a clinical pharmacist at KHCC."),
    HumanMessage(content="What is the standard dose of imatinib for CML?")
]

# SOLUTION: Invoke the LLM with these messages
response = llm.invoke(messages)
print("Approach A:", response.content[:200])

print("\n" + "="*50 + "\n")

# --- Approach B: Reusable prompt template ---
# SOLUTION: Create a ChatPromptTemplate with variables
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a clinical pharmacist at KHCC."),
    ("human", "What is the standard dose of {drug_name} for {condition}?")
])

# SOLUTION: Invoke the prompt with variables, then pass to LLM
formatted = prompt.invoke({"drug_name": "rituximab", "condition": "diffuse large B-cell lymphoma"})
response = llm.invoke(formatted)
print("Approach B:", response.content[:200])

---
## Cell 3 — Structured Output with LangChain

In Session 3, you used Pydantic models with OpenAI's `response_format`:
```python
response = client.beta.chat.completions.parse(
    model="gpt-4o-mini",
    response_format=PatientSummary,
    messages=[...]
)
```

LangChain makes this even simpler with `.with_structured_output()`.

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal

# Define a structured output schema
class DrugInfo(BaseModel):
    """Structured information about a medication."""
    drug_name: str = Field(description="Generic name of the drug")
    drug_class: str = Field(description="Pharmacological class")
    common_indication: str = Field(description="Primary clinical indication")
    route: Literal["oral", "IV", "subcutaneous", "topical", "other"] = Field(
        description="Primary route of administration"
    )

# SOLUTION: Create a structured LLM using .with_structured_output()
structured_llm = llm.with_structured_output(DrugInfo)

# SOLUTION: Invoke the structured LLM
result = structured_llm.invoke("Tell me about rituximab")

print(f"Drug: {result.drug_name}")
print(f"Class: {result.drug_class}")
print(f"Indication: {result.common_indication}")
print(f"Route: {result.route}")
print(f"\nType: {type(result)}")

**Key advantage:** The result is already a validated Pydantic object — no JSON parsing needed!

---
## Cell 4 — Runnables and Pipe Composition

LangChain's superpower is **composition** — you can chain steps together using the `|` (pipe) operator, just like Unix pipes.

Each step is a **Runnable** that has `.invoke()`, `.batch()`, and `.stream()` methods.

In [ ]:
from langchain_core.output_parsers import StrOutputParser

# Create a prompt template
clinical_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an oncology pharmacist at KHCC. Be concise."),
    ("human", "Summarize the key side effects of {drug_name} in 3 bullet points.")
])

# SOLUTION: Build a chain using the pipe operator
chain = clinical_prompt | llm | StrOutputParser()

# SOLUTION: Invoke the chain with a drug name
result = chain.invoke({"drug_name": "cisplatin"})

print("Chain result (string):")
print(result)

print("\n" + "="*50)
print("\nType:", type(result))  # Now it's a plain string, not an AIMessage

**Why pipes matter:**
- Each step transforms data and passes it to the next
- `prompt` turns variables into messages
- `llm` turns messages into an AIMessage
- `StrOutputParser()` extracts the text string
- The whole chain gets `.invoke()`, `.batch()`, `.stream()` for free

---
## Cell 5 — Create an Agent with a Simple Tool

In Session 3, you built a manual tool-calling loop. Now let's see how LangGraph's `create_react_agent` handles all of that automatically.

In [ ]:
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent

# Define a simple tool using the @tool decorator
@tool
def get_drug_formulary_status(drug_name: str) -> str:
    """Check if a drug is on the KHCC formulary and return its status."""
    # Mock formulary database
    formulary = {
        "imatinib": "On formulary - First-line CML therapy",
        "rituximab": "On formulary - Restricted to hematology",
        "pembrolizumab": "On formulary - Requires prior authorization",
        "nivolumab": "On formulary - Requires prior authorization",
        "aspirin": "On formulary - Unrestricted"
    }
    status = formulary.get(drug_name.lower(), "Not found on formulary")
    return f"{drug_name}: {status}"

# SOLUTION: Create an agent using create_react_agent
agent = create_react_agent(model=llm, tools=[get_drug_formulary_status])

# SOLUTION: Invoke the agent with a question that requires the tool
result = agent.invoke(
    {"messages": [{"role": "user", "content": "Is pembrolizumab on the KHCC formulary?"}]}
)

# Print all messages to see the agent loop
for msg in result["messages"]:
    role = msg.__class__.__name__
    print(f"\n[{role}]")
    if hasattr(msg, 'tool_calls') and msg.tool_calls:
        for tc in msg.tool_calls:
            print(f"  Tool call: {tc['name']}({tc['args']})")
    if msg.content:
        print(f"  {msg.content[:300]}")

**Compare with Session 3:** In Session 3 Lab 3, you built the tool-calling loop manually:
```python
while True:
    response = client.chat.completions.create(...)
    if response.choices[0].message.tool_calls:
        # Manually call the function
        # Manually append tool results
        # Loop again
    else:
        break
```

With `create_react_agent`, all of that is handled automatically!

---
## Stretch Challenge

1. Add a second tool `get_drug_interactions(drug_name: str) -> str` that returns mock interaction data
2. Rebuild the agent with both tools
3. Ask: "Is pembrolizumab on formulary and does it have any interactions?"
4. Observe how the agent calls both tools

### KHCC Connection
At KHCC, a formulary lookup agent could:
- Help pharmacists quickly check drug availability
- Flag restricted medications that need prior authorization
- Reduce medication errors by providing real-time formulary status
- Connect to the actual KHCC formulary database via a tool

In [ ]:
# STRETCH SOLUTION

@tool
def get_drug_interactions(drug_name: str) -> str:
    """Check for known drug interactions for a given medication."""
    interactions = {
        "pembrolizumab": "Caution with corticosteroids (may reduce efficacy). Avoid live vaccines.",
        "imatinib": "CYP3A4 interactions: avoid ketoconazole, rifampin. Monitor with warfarin.",
        "rituximab": "Avoid live vaccines. Caution with cisplatin (increased renal toxicity risk)."
    }
    info = interactions.get(drug_name.lower(), "No interaction data available.")
    return f"Interactions for {drug_name}: {info}"

# Rebuild agent with both tools
agent_v2 = create_react_agent(
    model=llm,
    tools=[get_drug_formulary_status, get_drug_interactions]
)

result = agent_v2.invoke(
    {"messages": [{"role": "user", "content": "Is pembrolizumab on formulary and does it have any interactions?"}]}
)

for msg in result["messages"]:
    role = msg.__class__.__name__
    print(f"\n[{role}]")
    if hasattr(msg, 'tool_calls') and msg.tool_calls:
        for tc in msg.tool_calls:
            print(f"  Tool call: {tc['name']}({tc['args']})")
    if msg.content:
        print(f"  {msg.content[:300]}")